### setup

In [1]:
# !pip install -U sentence-transformers
# !pip install datasets
# !pip install tensorflow_ranking

In [2]:
import pickle
import numpy as np
import os
import tqdm
import torch
import gc

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

from importlib import reload
from matplotlib import pyplot as plt

import matrix_approx_zeshel

matrix_approx_zeshel = reload(matrix_approx_zeshel)

!ls  | grep matrix_approx_zeshel

2024-03-28 17:47:27.213212: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/shevkunov/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


matrix_approx_zeshel.py


In [3]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc

import matplotlib.pyplot as plt


def load(limit, raw_path = "stand/log.local.logtime2.txt", path = "log.local.logtime2.bin", key_games = None, seed = 17, det_attempts = 0, key_size = 100):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = key_size, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            
            self.key_size = key_size
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            self.relevs = dict()
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def get_kmeans_games(self, all_from_labels=True):
            X = self.get_relevs("train").T

            k_func = (
                (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
                if not all_from_labels else
                (lambda C : from_labels(X, C.labels_))
            )
            K_KMeans = k_func(
                KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
            )

            return K_KMeans

        def set_kmeans_games_as_key(self, *args, **kwargs):
            self.key_games = self.get_kmeans_games(*args, **kwargs)
            return self

        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id][:self.key_size]
                    for r_i in r_i_array[:self.key_size]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_relevs(self, t = "train"):
            if t not in self.relevs:
                self.relevs[t] = np.array([
                    np.array([g_i[1] for g_i in r[2]])
                    for r in self.get_requests(t)
                ])
                
            return self.relevs[t]

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

In [4]:
def load_ment_to_ent_scores(directory = "yugioh", shuffle_rows = 0, full = True):
    data = list()

    for file in os.listdir(directory):
        path = f"{directory}/{file}"
        print(f"Loading file {path}")
        with open(path, "rb") as f:
            data.append(
                pickle.load(f)
            )
    data = sorted(data, key = lambda x: x["arg_dict"]["n_ment_start"])

    for i in range(len(data) - 1):
        if full:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] == data[i + 1]["arg_dict"]["n_ment_start"]
        else:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] <= data[i + 1]["arg_dict"]["n_ment_start"]
        
    ment_to_ent_scores = list(map(lambda x: x["ment_to_ent_scores"], data))
    ment_to_ent_scores = np.vstack(ment_to_ent_scores)
    print("Loaded shape = ", ment_to_ent_scores.shape)
    
    if shuffle_rows:
        print(f"Shuffling... (seed = {shuffle_rows})")
        np.random.seed(shuffle_rows)
        np.random.shuffle(ment_to_ent_scores)
    
    return ment_to_ent_scores

In [5]:
from sklearn.cluster import KMeans
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances


def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

class EvalContextRelevs:
    def __init__(self, relevs, key_size = 100, train_size = 0.7, key_games = None, seed = 17, shuffle=False, det_attempts = 0):
        self.relevs = np.array(relevs)
        self.reqs_count = self.relevs.shape[0]
        self.games_count = self.relevs.shape[1]
        
        self.key_size = key_size
        self.key_games = (
            np.random.choice(np.arange(self.games_count), key_size, replace=False)
            if key_games is None else
            key_games
        )
        np.random.seed(seed)
        if shuffle:
            np.random.shuffle(self.relevs)

        self.try_det_attempts(det_attempts)
        self.train_split = int(self.reqs_count * train_size)

        assert key_size + 1 < self.train_split


        self.key_relevs = self.relevs[:key_size]
        self.train_relevs = self.relevs[key_size + 1: self.train_split]
        self.test_relevs = self.relevs[self.train_split:]

        self.slices = ["key", "train", "test"]
        print(len(self.key_relevs), len(self.train_relevs), len(self.test_relevs))

    def get_top_games(self):
        return self.relevs.mean(axis=0).argsort()[:self.key_size]

    def set_top_games_as_key(self):
        self.key_games = self.get_top_games()
        return self

    def get_kmeans_games(self, all_from_labels=True):
        X = self.get_relevs("train").T

        k_func = (
            (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
            if not all_from_labels else
            (lambda C : from_labels(X, C.labels_))
        )
        K_KMeans = k_func(
            KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
        )
        
        return K_KMeans
    
    def set_kmeans_games_as_key(self, *args, **kwargs):
        self.key_games = self.get_kmeans_games(*args, **kwargs)
        return self
    

    def try_det_attempts(self, det_attempts):
        def get_det(r):
            kr = r[:self.key_size, self.key_games] - r.mean()
            return np.abs(np.linalg.det(kr))

        best_i_array = np.arange(len(self.relevs))

        for i in range(det_attempts):

            r_i_array = np.arange(len(self.relevs))
            np.random.shuffle(r_i_array)

            n, o = get_det(self.relevs[r_i_array, :]), get_det(self.relevs[best_i_array, :])
            
            # print(f"try update key_reqs ({o} vs {n}...")
            if n > o:
                best_i_array = r_i_array
                print(f"updated det ({i}, {o} -> {n})")

        print("Best det = ", get_det(self.relevs[best_i_array, :]))

        self.relevs = self.relevs[best_i_array, :]
        print("Current de = ", get_det(self.relevs))
        
    def get_relevs(self, t = "train"):
        if t == "train":
            return self.train_relevs
        elif t == "key":
            return self.key_relevs
        elif t == "test":
            return self.test_relevs
        else:
            assert False
            
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

In [6]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        self.trus_top = dict()
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):    
        recs = self.recommend(t)
        
        if isinstance(recs, list):
            recs = np.array(recs)
        
        mse = np.mean((recs - self.get_user_scores(t)) ** 2)
        
        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :topsize].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :topsize]
        
        if (t, topsize) not in self.trus_top:
            trus = self.get_user_scores(t)
            trus = np.argsort(-trus, axis=1)[:, :topsize]
            self.trus_top[(t, topsize)] = trus
            
        trus = self.trus_top[(t, topsize)]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)
    
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "matrix_l2": 0,
    "dssm_l2": 0,
    "train_popbias": False,
    "train_bias": False,
    "train_vbias": False,
    "verbose": True,
    "train_matrix": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
    "activation": "relu",
    "score_verbose": 0,
    "trainable_items": False,
    "use_keys_in_train": False,
    "Winit": "glorot0.01",
    "Wfreeze": False,
    "TEinit": "legacy",
    "normalize_embs": True,
    "save_callback": None,
    "inner_dim": None
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(fit_kwargs)
        self.fit_kwargs = p
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[i] for i in sorted(ctx.key_games)])
                    for r_i in ctx.get_relevs(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[g_i] for r in ctx.get_relevs("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        
        if self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["TEinit"] == "legacy":
                value = self.embed_games
            elif self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                value = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print(type(value))
            else:
                assert False, "unknown TEinit: " + str(self.fit_kwargs["TEinit"])
                
            self.trainable_games = tf.Variable(
                tf.convert_to_tensor(value, dtype=tf.float32),
                trainable=True
            )
        else:
            if self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                self.embed_games = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print("ANNCur init")
            
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = ctx.get_relevs("key")
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
        
        self.slice2best = {
            t : 0
            for t in ctx.slices
        }
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        if self.fit_kwargs["normalize_embs"]:
            return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        else:
            return self.embed_users[t]
    
    def get_game_embs(self):
        if not self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["normalize_embs"]:
                return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
            else:
                return self.embed_games 
        else:
            return self.trainable_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.p = p
        self.train_bias = p["train_bias"]
        self.train_vbias = p["train_vbias"]
        self.train_popbias = p["train_popbias"]
        self.train_matrix = p["train_matrix"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        if p["use_keys_in_train"]:
            train_user_scores = np.vstack([
                self.get_user_scores("key"),
                self.get_user_scores("train")
            ])
            train_user_embs = np.vstack([
                self.get_user_embs("key"),
                self.get_user_embs("train")
            ])
        else:
            train_user_scores = self.get_user_scores("train")
            train_user_embs = self.get_user_embs("train")

        game_embs = self.get_game_embs()
        
        
        
        if self.p["Winit"] == "glorot0.01":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
            values = values / 100.  
        elif self.p["Winit"] == "glorot":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        elif self.p["Winit"] == "eye":
            values = np.eye(train_user_embs.shape[1], game_embs.shape[1])
        else:
            assert False, "init is not known:" + str(self.p["init"])
            
        self.W = tf.Variable(values, trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            inner_dim = p["inner_dim"] if p.get("inner_dim", None) is not None else dim
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(inner_dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(inner_dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        if self.train_vbias:
            self.vb = tf.Variable(
                np.zeros_like(self.game_avg_scores["train"], dtype=np.float32),
                trainable = True
            ) 
        
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = (
                            game_embs[game_slice]
                            if isinstance(game_embs, np.ndarray) else
                            tf.gather(game_embs, game_slice)
                        )
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = 0
                    
                    if self.train_matrix:
                        logits += train_user_embs_ @ self.W @ tf.transpose(game_embs_)

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_vbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.vb
                        else:
                            logits += tf.gather(self.vb, game_slice)
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_signed":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    sf = tf.nn.softmax(logits, axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        sf,
                        -sf
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_sum(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                elif self.loss == "KL100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    mask = np.argsort(-scores, axis=1) < 100
                    v = tf.keras.losses.KLDivergence()((scores.T >= qs.T).T, logits)
                else:
                    assert False
                    
                if self.p["c"]:
                    v += tf.reduce_mean(self.W * self.W) * p["c"]
                    
                if self.p["dssm_l2"]:
                    for weights_ in [self.u_dssm.weights, self.g_dssm.weights]:
                        for weight_ in weights_:
                            v += tf.reduce_sum(weight_ * weight_) * self.p["dssm_l2"]
                
                if self.p["matrix_l2"]:
                    v += tf.reduce_sum(self.W * self.W) * self.p["matrix_l2"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            if self.train_matrix and (not self.p["Wfreeze"]):
                weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_vbias:
                weights += [self.vb]   
                
            if self.train_bias:
                weights += [self.b]   
                
            
            if self.p["trainable_items"]:
                weights += [self.trainable_games]
                
                
            opt.minimize(loss, weights)
            
            if p["score_verbose"] and (i % p["score_verbose"] == 0):
                print(f"\n=== Iteration {i} ===")
                score = dict()
                for sl in self.ctx.slices:
                    score[sl] = self.get_score(sl)
                    print(f"slice = {sl}, score = {score[sl]}")
                    
                if score["train"] > self.slice2best["train"]:
                    self.slice2best = score
                    if p["save_callback"] is not None:
                        p["save_callback"](self)
                
        print("last loss = ", loss())

    def recommend(self, t):
        logits = 0
                    
        if self.train_matrix:
            logits += self.get_user_embs(t) @ self.W @ tf.transpose(self.get_game_embs())

        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_vbias:
            logits += self.vb
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))
            
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))
            
class AnnCUR(Popular):
    def __init__(self, ctx, oracle=False, key_games=None, name=None):
        super().__init__(ctx)
        self.oracle = oracle
        self.ctx = ctx
        self.name = name

        self.key_cols_idx = np.array(sorted(ctx.key_games if key_games is None else key_games))
        rows_idx = np.arange(ctx.get_relevs("train").shape[0])
        
        self.cur = matrix_approx_zeshel.CURApprox(
            rows=torch.from_numpy(ctx.get_relevs("train")),
            cols=torch.from_numpy(ctx.get_relevs("train")[:, self.key_cols_idx]),
            row_idxs=rows_idx,
            col_idxs=self.key_cols_idx,
            approx_preference="rows",
            A=(torch.from_numpy(ctx.get_relevs("train")) if oracle else None)
        )        
        
    def __repr__(self):
        if self.name is None:
            return f"AnnCur({id(self)})"
        else:
            return f"AnnCur({self.name} - {id(self)})"
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        return self.cur.latent_cols.T

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        key_relevs = torch.from_numpy(self.ctx.get_relevs(t)[:, self.key_cols_idx])
        return self.cur.get_complete_row(key_relevs)
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

In [7]:
def coitem_algorithm(n_support, candidate_items, target_items, min_item_rel_norm=1e-5, eps=1e-6):
    support_items = []
    support_items_scores = []
    n_c, n_q = candidate_items.shape
    n_t = target_items.shape[0]

    candidate_item_squared_norms = (candidate_items ** 2).sum(axis=1)
    min_item_norm = min_item_rel_norm * np.sqrt(candidate_item_squared_norms.mean())
    
    candidate_coitems = candidate_items.dot(
        target_items.T.dot(target_items)
    )
    orthonormed_support_items = np.zeros((n_support, n_q))
    remaining_items = np.ones(candidate_items.shape[0], dtype="bool")
    for t in tqdm.tqdm_notebook(range(n_support)):
        scores = (candidate_coitems * candidate_items).sum(axis=1)
        remaining_items &= (candidate_item_squared_norms >= min_item_norm ** 2)
        scores[remaining_items] /= candidate_item_squared_norms[remaining_items]
        scores[~remaining_items] = 0
        
        new_item_id = np.argmax(scores)
        assert remaining_items[new_item_id]
        support_items.append(new_item_id)
        support_items_scores.append(scores[new_item_id] / (n_t * n_q))
        remaining_items[new_item_id] = False
        
        new_item = candidate_items[new_item_id].copy()
        new_item -= orthonormed_support_items[:t].T.dot(
            orthonormed_support_items[:t].dot(new_item)
        )
        assert np.allclose((new_item ** 2).sum(), candidate_item_squared_norms[new_item_id], atol=eps)
        new_item /= np.sqrt((new_item ** 2).sum())
        orthonormed_support_items[t] = new_item
        new_coitem = candidate_coitems[new_item_id] / np.sqrt(candidate_item_squared_norms[new_item_id])
        
        coefs = (candidate_items * new_item).sum(axis=1)
        candidate_item_squared_norms -= coefs ** 2
        assert np.all(candidate_item_squared_norms[remaining_items] > -eps)
        
        candidate_coitems -= coefs.reshape((-1, 1)).dot(new_coitem.reshape((1, -1)))
        cocoefs = (candidate_coitems * new_item).sum(axis=1, keepdims=True)
        candidate_coitems -= cocoefs.dot(new_item.reshape((1, -1)))
    return np.array(support_items, dtype=np.int64), np.array(support_items_scores)

# Preparate

In [8]:
def do(ctx, name, coitem=True, kmeans=True, top=True, random=True):
    if coitem:
        t = ctx.get_relevs("train").T
        t = (t - t.mean()) / t.std()  # like in other comparisons with RBE
        ctx.key_games = list(coitem_algorithm(ctx.key_size, t, t, 1e-8, eps=1e9)[0])

        ma = AnnCUR(ctx, key_games=ctx.key_games)

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")

        GE = ma.get_game_embs()
        QE = torch.from_numpy(ma.ctx.get_relevs("test")[:, ma.key_cols_idx])

        fname = f"./GE_QE_AnnCURxCoItem_{name}_{str(round(te, 4))[2:]}.npz"
        np.savez_compressed(fname, QE=QE.numpy(), GE=GE.numpy())
        print(fname)
        
    if kmeans:
        ctx.set_kmeans_games_as_key()
        ma = AnnCUR(ctx, key_games=ctx.key_games)

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")

        GE = ma.get_game_embs()
        QE = torch.from_numpy(ma.ctx.get_relevs("test")[:, ma.key_cols_idx])

        fname = f"./GE_QE_AnnCURxKMeans_{name}_{str(round(te, 4))[2:]}.npz"
        np.savez_compressed(fname, QE=QE.numpy(), GE=GE.numpy())
        print(fname)
        
    if top:
        ctx.set_top_games_as_key()
        ma = AnnCUR(ctx, key_games=ctx.key_games)

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")

        GE = ma.get_game_embs()
        QE = torch.from_numpy(ma.ctx.get_relevs("test")[:, ma.key_cols_idx])

        fname = f"./GE_QE_AnnCURxTop_{name}_{str(round(te, 4))[2:]}.npz"
        np.savez_compressed(fname, QE=QE.numpy(), GE=GE.numpy())
        print(fname)
        
    if random:
        ctx.key_games = np.random.choice(np.arange(ctx.games_count), ctx.key_size, replace=False)
        ma = AnnCUR(ctx, key_games=ctx.key_games)

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")

        GE = ma.get_game_embs()
        QE = torch.from_numpy(ma.ctx.get_relevs("test")[:, ma.key_cols_idx])

        fname = f"./GE_QE_AnnCURxRandom_{name}_{str(round(te, 4))[2:]}.npz"
        np.savez_compressed(fname, QE=QE.numpy(), GE=GE.numpy())
        print(fname)

In [9]:
m_ = None
def do_RBE(ctx, name, coitem=True, kmeans=True, top=True, random=True, N = 100000, LR = 0.0005, train_matrix=True, inner_dim=None):
    def get_save_callback(t, fname):
        def save_callback(m):
            global m_
            m_ = m
            GE_ = m.get_game_embs()
            GE = np.hstack([
                GE_,
                m.g_dssm(GE_),
                m.vb.numpy().reshape((-1, 1)) ,  # Vertical bias
                ctx.get_relevs("train").mean(axis=0).reshape((-1, 1))  # popbias
            ])

            R_All = ctx.get_relevs(t)
            QE_ = m.get_user_embs(t)
            QE = np.hstack([
                QE_ @ m.W,
                m.u_dssm(QE_),
                np.ones((R_All.shape[0], 1)),
                np.ones((R_All.shape[0], 1)) * m.pb
            ])
            
            l, r = fname.split(".npz")
            
            train_score = m.get_score("train")
            score = m.get_score(t)
            fname_ = l + f"_{str(round(train_score, 4))}_{str(round(score, 4))}.npz" + r
            print(f"Saving {fname_} ...")
            np.savez_compressed(fname_, QE=QE, GE=GE)
        return save_callback
            
    if coitem:
        t = ctx.get_relevs("train").T
        t = (t - t.mean()) / t.std()  # like in other comparisons with RBE
        ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': train_matrix,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "inner_dim": inner_dim,
            "save_callback": get_save_callback("test", f"./GE_QE_RBExCoItem_{name}.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)
        
    if kmeans:
        ctx.set_kmeans_games_as_key()
        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': train_matrix,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "inner_dim": inner_dim,
            "save_callback": get_save_callback("test", f"./GE_QE_RBExKMeans_{name}.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)
        
    if top:
        ctx.set_top_games_as_key()
        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': train_matrix,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "inner_dim": inner_dim,
            "save_callback": get_save_callback("test", f"./GE_QE_RBExTop_{name}.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)
        
    if random:
        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': train_matrix,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "inner_dim": inner_dim,
            "save_callback": get_save_callback("test", f"./GE_QE_{name}_RBExRandom.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)

In [10]:
def get_Rp_basic(GE, A, R, QE_i, L):
    Ae = GE[A]

    u = np.linalg.pinv(Ae.T @ Ae) @ Ae.T @ R
    assert u.shape == QE_i.shape
    Rp = GE @ (u * L + QE_i * (1 - L))
    
    return Rp


def do_AXN(ctx, loaded, get_Rp, Z="test", STRIP = 0.05, Ls=np.linspace(0, 1, 11), Ks=[25], T_TS_s = [(200, 100)], not_deduplicate = False):
    # q = ctx.get_requests(Z)
    R_All = ctx.get_relevs(Z)

    GE, QE = loaded["GE"], loaded["QE"]

    best = 0
    best_arg = None

    for randomFirst in [False]:
        for K in Ks:
            for L in Ls:
                for T, TS in T_TS_s:
                    results = list()


                    As = list()
                    for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                        A = (
                            ctx.key_games[:K]
                            if randomFirst else
                            (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                        )
                        Rt = R_All[i]
                        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                        for _ in range(T // K - 1):
                            Rp = get_Rp(GE, A, Rt[A], QE[i], L)

                            An = list(A)
                            for ai in Rp.argsort()[::-1]:
                                if not_deduplicate or (ai not in An):
                                    An.append(ai)
                                if len(An) == len(A) + K:
                                    break
                            A = np.array(An)

                        assert len(A) == T
                        As.append(A)


                        A = sorted([
                            (-Rt[ai], ai)
                            for ai in A
                        ])[:TS]
                        A = [ai for _, ai in A]

                        pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                        results.append(ev(pred_top, gt_top) / float(TS))

                    if np.mean(results) > best:
                        best = np.mean(results)
                        best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                        best_arg_d = {
                            "K": K,
                            "L": L,
                            "randomFirst": randomFirst,
                            "score": np.mean(results)
                        }
                    print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)} best = {best}")

    return best_arg

## RecSys LT

In [ ]:
L = 7000
N = 1000
DA = 50

In [ ]:
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA, key_size=100)

In [ ]:
do_RBE(ctx, "RecSysLT_noW_", N=300000, kmeans=False, top=False, random=False, train_matrix=False)

## RecSys PairWise

In [ ]:
del ctx
gc.collect()

In [ ]:
L = 7000
N = 1000
DA = 50


ctx = load(L, raw_path = "//dev/null/stand/log.local.txt",
           path="log.local.bin.old", seed=17, det_attempts=DA, key_size=100)
print("LOADED")

In [ ]:
do_RBE(ctx, "RecSysLT_noW_", N=300000, kmeans=False, top=False, random=False, train_matrix=False)

# ZESHEL

In [12]:
DATASETS = ["yugioh", "pro_wrestling", "star_trek", "doctor_who", "military"]

In [12]:
del ctx

NameError: name 'ctx' is not defined

In [ ]:
gc.collect()

In [ ]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100,
        key_size=100
    )

    print("LOADED")
    
    do_RBE(ctx, "RecSysLT_noW_", N=200000, kmeans=False, top=False, random=False, train_matrix=False)
    
    del ctx
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
LOADED


/var/tmp/ipykernel_3627610/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10031)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2260, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.022000000000000002 tf.Tensor(43.26422, shape=(), dtype=float32) 100
slice = key, score = 0.022000000000000002
np.mean(results), mse, len(results) =  0.023769911504424777 tf.Tensor(42.77875, shape=(), dtype=float32) 2260
slice = train, score = 0.023769911504424777
np.mean(results), mse, len(results) =  0.025468904244817372 tf.Tensor(42.439697, shape=(), dtype=float32) 1013
slice = test, score = 0.025468904244817372
np.mean(results), mse, len(results) =  0.023769911504424777 tf.Tensor(42.77875, shape=(), dtype=float32) 2260
np.mean(results), mse, len(results) =  0.025468904244817372 tf.Tensor(42.439697, shape=(), dtype=float32) 1013
Saving ./GE_QE_RBExCoItem_RecSysLT_noW__0.0

np.mean(results), mse, len(results) =  0.6093628318584071 tf.Tensor(1107.9617, shape=(), dtype=float32) 2260
slice = train, score = 0.6093628318584071
np.mean(results), mse, len(results) =  0.5633464955577493 tf.Tensor(1097.3492, shape=(), dtype=float32) 1013
slice = test, score = 0.5633464955577493
np.mean(results), mse, len(results) =  0.6093628318584071 tf.Tensor(1107.9617, shape=(), dtype=float32) 2260
np.mean(results), mse, len(results) =  0.5633464955577493 tf.Tensor(1097.3492, shape=(), dtype=float32) 1013
Saving ./GE_QE_RBExCoItem_RecSysLT_noW__0.6094_0.5633.npz ...

=== Iteration 12000 ===
np.mean(results), mse, len(results) =  0.5987 tf.Tensor(1297.4491, shape=(), dtype=float32) 100
slice = key, score = 0.5987
np.mean(results), mse, len(results) =  0.6086548672566371 tf.Tensor(1251.7366, shape=(), dtype=float32) 2260
slice = train, score = 0.6086548672566371
np.mean(results), mse, len(results) =  0.5628825271470879 tf.Tensor(1247.06, shape=(), dtype=float32) 1013
slice = test

...

np.mean(results), mse, len(results) =  0.4169198459922996 tf.Tensor(52902.434, shape=(), dtype=float32) 2857
slice = train, score = 0.4169198459922996
np.mean(results), mse, len(results) =  0.37419227738376676 tf.Tensor(52190.92, shape=(), dtype=float32) 1269
slice = test, score = 0.37419227738376676

=== Iteration 120000 ===
np.mean(results), mse, len(results) =  0.3864000000000001 tf.Tensor(54292.54, shape=(), dtype=float32) 100
slice = key, score = 0.3864000000000001
np.mean(results), mse, len(results) =  0.41739236961848103 tf.Tensor(53936.793, shape=(), dtype=float32) 2857
slice = train, score = 0.41739236961848103
np.mean(results), mse, len(results) =  0.3744365642237983 tf.Tensor(53132.445, shape=(), dtype=float32) 1269
slice = test, score = 0.3744365642237983

=== Iteration 121000 ===
np.mean(results), mse, len(results) =  0.3856 tf.Tensor(55155.066, shape=(), dtype=float32) 100
slice = key, score = 0.3856
np.mean(results), mse, len(results) =  0.41737136856842855 tf.Tensor(549


=== Iteration 137000 ===
np.mean(results), mse, len(results) =  0.38670000000000004 tf.Tensor(68612.82, shape=(), dtype=float32) 100
slice = key, score = 0.38670000000000004
np.mean(results), mse, len(results) =  0.4178333916695835 tf.Tensor(68657.62, shape=(), dtype=float32) 2857
slice = train, score = 0.4178333916695835
np.mean(results), mse, len(results) =  0.37496453900709226 tf.Tensor(67918.97, shape=(), dtype=float32) 1269
slice = test, score = 0.37496453900709226

=== Iteration 138000 ===
np.mean(results), mse, len(results) =  0.38660000000000005 tf.Tensor(69317.445, shape=(), dtype=float32) 100
slice = key, score = 0.38660000000000005
np.mean(results), mse, len(results) =  0.4192194609730487 tf.Tensor(70004.94, shape=(), dtype=float32) 2857
slice = train, score = 0.4192194609730487
np.mean(results), mse, len(results) =  0.37535855003940116 tf.Tensor(69390.68, shape=(), dtype=float32) 1269
slice = test, score = 0.37535855003940116

=== Iteration 139000 ===
np.mean(results), mse

np.mean(results), mse, len(results) =  0.41504025201260064 tf.Tensor(83140.49, shape=(), dtype=float32) 2857
slice = train, score = 0.41504025201260064
np.mean(results), mse, len(results) =  0.37286052009456266 tf.Tensor(82302.23, shape=(), dtype=float32) 1269
slice = test, score = 0.37286052009456266

=== Iteration 154000 ===
np.mean(results), mse, len(results) =  0.3829000000000001 tf.Tensor(85148.375, shape=(), dtype=float32) 100
slice = key, score = 0.3829000000000001
np.mean(results), mse, len(results) =  0.41638431921596086 tf.Tensor(84354.82, shape=(), dtype=float32) 2857
slice = train, score = 0.41638431921596086
np.mean(results), mse, len(results) =  0.374468085106383 tf.Tensor(83348.92, shape=(), dtype=float32) 1269
slice = test, score = 0.374468085106383

=== Iteration 155000 ===
np.mean(results), mse, len(results) =  0.3882000000000001 tf.Tensor(85826.85, shape=(), dtype=float32) 100
slice = key, score = 0.3882000000000001
np.mean(results), mse, len(results) =  0.4163423171


=== Iteration 171000 ===
np.mean(results), mse, len(results) =  0.38939999999999997 tf.Tensor(99304.516, shape=(), dtype=float32) 100
slice = key, score = 0.38939999999999997
np.mean(results), mse, len(results) =  0.4189429471473574 tf.Tensor(99231.06, shape=(), dtype=float32) 2857
slice = train, score = 0.4189429471473574
np.mean(results), mse, len(results) =  0.37438928289992124 tf.Tensor(98236.77, shape=(), dtype=float32) 1269
slice = test, score = 0.37438928289992124

=== Iteration 172000 ===
np.mean(results), mse, len(results) =  0.3872 tf.Tensor(101012.13, shape=(), dtype=float32) 100
slice = key, score = 0.3872
np.mean(results), mse, len(results) =  0.41881694084704235 tf.Tensor(100389.83, shape=(), dtype=float32) 2857
slice = train, score = 0.41881694084704235
np.mean(results), mse, len(results) =  0.37426319936958236 tf.Tensor(99401.414, shape=(), dtype=float32) 1269
slice = test, score = 0.37426319936958236

=== Iteration 173000 ===
np.mean(results), mse, len(results) =  0.3


=== Iteration 188000 ===
np.mean(results), mse, len(results) =  0.386 tf.Tensor(114352.38, shape=(), dtype=float32) 100
slice = key, score = 0.386
np.mean(results), mse, len(results) =  0.41895344767238357 tf.Tensor(114200.02, shape=(), dtype=float32) 2857
slice = train, score = 0.41895344767238357
np.mean(results), mse, len(results) =  0.37441292356185973 tf.Tensor(113176.43, shape=(), dtype=float32) 1269
slice = test, score = 0.37441292356185973


In [ ]:
for dataset in DATASETS[2:]:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100,
        key_size=100
    )

    print("LOADED")
    
    do_RBE(ctx, f"{dataset}_noW_", N=200000, kmeans=False, top=False, random=False, train_matrix=False)
    
    del ctx
    gc.collect()




DATASET =  star_trek
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False2800.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False2200.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False3200.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False1000.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False200.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False2400.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False1400.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False2000.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False800.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False600.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False3000.pkl
Loading file star_trek/ment_to_ent_scores_n_

/var/tmp/ipykernel_4060/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

self.embed_users['train'].shape =  (2857, 100)
self.embed_games.shape =  (34430, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 34430)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2857, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.019500000000000003 tf.Tensor(41.748695, shape=(), dtype=float32) 100
slice = key, score = 0.019500000000000003
np.mean(results), mse, len(results) =  0.013563178158907946 tf.Tensor(40.736134, shape=(), dtype=float32) 2857
slice = train, score = 0.013563178158907946
np.mean(results), mse, len(results) =  0.013144208037825058 tf.Tensor(40.783688, shape=(), dtype=float32) 1269
slice = test, score = 0.013144208037825058
np.mean(results), mse, len(results) =  0.013563178158907946 tf.Tensor(40.736134, shape=(), dtype=float32) 2857
np.mean(results), mse, len(results) =  0.013144208037825058 tf.Tensor(40.783688, shape=(), dtype=float32) 1269
Saving ./GE_QE_RBExCoItem_star_trek_noW_

np.mean(results), mse, len(results) =  0.3952432621631082 tf.Tensor(1024.8191, shape=(), dtype=float32) 2857
slice = train, score = 0.3952432621631082
np.mean(results), mse, len(results) =  0.3635224586288416 tf.Tensor(1016.44543, shape=(), dtype=float32) 1269
slice = test, score = 0.3635224586288416

=== Iteration 14000 ===
np.mean(results), mse, len(results) =  0.3614 tf.Tensor(1169.3523, shape=(), dtype=float32) 100
slice = key, score = 0.3614
np.mean(results), mse, len(results) =  0.39371718585929294 tf.Tensor(1142.0992, shape=(), dtype=float32) 2857
slice = train, score = 0.39371718585929294
np.mean(results), mse, len(results) =  0.36205673758865253 tf.Tensor(1136.4113, shape=(), dtype=float32) 1269
slice = test, score = 0.36205673758865253

=== Iteration 15000 ===
np.mean(results), mse, len(results) =  0.3635 tf.Tensor(1240.1537, shape=(), dtype=float32) 100
slice = key, score = 0.3635
np.mean(results), mse, len(results) =  0.39485124256212806 tf.Tensor(1212.4926, shape=(), dtype

...

self.embed_users['train'].shape =  (1579, 100)
self.embed_games.shape =  (104520, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 104520)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1579, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.009399999999999999 tf.Tensor(39.79522, shape=(), dtype=float32) 100
slice = key, score = 0.009399999999999999
np.mean(results), mse, len(results) =  0.012773907536415455 tf.Tensor(36.41825, shape=(), dtype=float32) 1579
slice = train, score = 0.012773907536415455
np.mean(results), mse, len(results) =  0.010541666666666666 tf.Tensor(36.46235, shape=(), dtype=float32) 720
slice = test, score = 0.010541666666666666
np.mean(results), mse, len(results) =  0.012773907536415455 tf.Tensor(36.41825, shape=(), dtype=float32) 1579
np.mean(results), mse, len(results) =  0.010541666666666666 tf.Tensor(36.46235, shape=(), dtype=float32) 720
Saving ./GE_QE_RBExCoItem_military_noW__0.012

np.mean(results), mse, len(results) =  0.29930968967701077 tf.Tensor(2626.6045, shape=(), dtype=float32) 1579
slice = train, score = 0.29930968967701077
np.mean(results), mse, len(results) =  0.2682361111111111 tf.Tensor(2620.134, shape=(), dtype=float32) 720
slice = test, score = 0.2682361111111111

=== Iteration 12000 ===
np.mean(results), mse, len(results) =  0.25960000000000005 tf.Tensor(3142.5544, shape=(), dtype=float32) 100
slice = key, score = 0.25960000000000005
np.mean(results), mse, len(results) =  0.3091070297656745 tf.Tensor(2856.7253, shape=(), dtype=float32) 1579
slice = train, score = 0.3091070297656745
np.mean(results), mse, len(results) =  0.2763333333333334 tf.Tensor(2849.149, shape=(), dtype=float32) 720
slice = test, score = 0.2763333333333334
np.mean(results), mse, len(results) =  0.3091070297656745 tf.Tensor(2856.7253, shape=(), dtype=float32) 1579
np.mean(results), mse, len(results) =  0.2763333333333334 tf.Tensor(2849.149, shape=(), dtype=float32) 720
Saving ./

np.mean(results), mse, len(results) =  0.3332678910702977 tf.Tensor(7644.4517, shape=(), dtype=float32) 1579
np.mean(results), mse, len(results) =  0.2913472222222222 tf.Tensor(7653.069, shape=(), dtype=float32) 720
Saving ./GE_QE_RBExCoItem_military_noW__0.3333_0.2913.npz ...

=== Iteration 26000 ===
np.mean(results), mse, len(results) =  0.269 tf.Tensor(8831.863, shape=(), dtype=float32) 100
slice = key, score = 0.269
np.mean(results), mse, len(results) =  0.3293603546548448 tf.Tensor(8212.299, shape=(), dtype=float32) 1579
slice = train, score = 0.3293603546548448
np.mean(results), mse, len(results) =  0.2883611111111111 tf.Tensor(8182.607, shape=(), dtype=float32) 720
slice = test, score = 0.2883611111111111

=== Iteration 27000 ===
np.mean(results), mse, len(results) =  0.27759999999999996 tf.Tensor(9421.39, shape=(), dtype=float32) 100
slice = key, score = 0.27759999999999996
np.mean(results), mse, len(results) =  0.3387840405319823 tf.Tensor(8763.959, shape=(), dtype=float32) 15


=== Iteration 41000 ===
np.mean(results), mse, len(results) =  0.2802 tf.Tensor(17900.686, shape=(), dtype=float32) 100
slice = key, score = 0.2802
np.mean(results), mse, len(results) =  0.3411779607346422 tf.Tensor(16928.555, shape=(), dtype=float32) 1579
slice = train, score = 0.3411779607346422
np.mean(results), mse, len(results) =  0.29468055555555556 tf.Tensor(16943.838, shape=(), dtype=float32) 720
slice = test, score = 0.29468055555555556

=== Iteration 42000 ===
np.mean(results), mse, len(results) =  0.2882 tf.Tensor(18644.246, shape=(), dtype=float32) 100
slice = key, score = 0.2882
np.mean(results), mse, len(results) =  0.3549081697276758 tf.Tensor(17665.598, shape=(), dtype=float32) 1579
slice = train, score = 0.3549081697276758
np.mean(results), mse, len(results) =  0.30541666666666667 tf.Tensor(17659.695, shape=(), dtype=float32) 720
slice = test, score = 0.30541666666666667
np.mean(results), mse, len(results) =  0.3549081697276758 tf.Tensor(17665.598, shape=(), dtype=flo

np.mean(results), mse, len(results) =  0.35169727675744145 tf.Tensor(30213.475, shape=(), dtype=float32) 1579
slice = train, score = 0.35169727675744145
np.mean(results), mse, len(results) =  0.30090277777777774 tf.Tensor(30262.83, shape=(), dtype=float32) 720
slice = test, score = 0.30090277777777774

=== Iteration 59000 ===
np.mean(results), mse, len(results) =  0.2831 tf.Tensor(32630.648, shape=(), dtype=float32) 100
slice = key, score = 0.2831
np.mean(results), mse, len(results) =  0.3470044331855605 tf.Tensor(31256.227, shape=(), dtype=float32) 1579
slice = train, score = 0.3470044331855605
np.mean(results), mse, len(results) =  0.30004166666666665 tf.Tensor(31340.098, shape=(), dtype=float32) 720
slice = test, score = 0.30004166666666665

=== Iteration 60000 ===
np.mean(results), mse, len(results) =  0.2826 tf.Tensor(33532.742, shape=(), dtype=float32) 100
slice = key, score = 0.2826
np.mean(results), mse, len(results) =  0.3462444585180494 tf.Tensor(32244.123, shape=(), dtype=fl

In [ ]:
Saving ./GE_QE_RBExCoItem_yugioh_noW__0.6455_0.5738.npz

Saving ./GE_QE_RBExCoItem_pro_wrestling_noW__0.6252_0.5203.npz ...
Saving ./GE_QE_RBExCoItem_pro_wrestling_noW__0.6369_0.517.npz ...

Saving ./GE_QE_RBExCoItem_star_trek_noW__0.4213_0.3782.npz ...

Saving ./GE_QE_RBExCoItem_doctor_who_noW__0.3347_0.2911.npz ..

Saving ./GE_QE_RBExCoItem_military_noW__0.3707_0.3246.npz ...
( ip)

In [13]:
for dataset in DATASETS[4:]:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100,
        key_size=100
    )

    print("LOADED")
    
    do_RBE(ctx, f"{dataset}_noW_", N=200000, kmeans=False, top=False, random=False, train_matrix=False)
    
    del ctx
    gc.collect()




DATASET =  military
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0500.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1500.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1300.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1600.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0800.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1700.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2300.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1100.pkl
Loading file military/ment_to_ent_scores_n

/var/tmp/ipykernel_2826/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

self.embed_users['train'].shape =  (1579, 100)
self.embed_games.shape =  (104520, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 104520)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1579, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.014800000000000002 tf.Tensor(41.165077, shape=(), dtype=float32) 100
slice = key, score = 0.014800000000000002
np.mean(results), mse, len(results) =  0.016985433818872703 tf.Tensor(39.854404, shape=(), dtype=float32) 1579
slice = train, score = 0.016985433818872703
np.mean(results), mse, len(results) =  0.013541666666666667 tf.Tensor(38.352516, shape=(), dtype=float32) 720
slice = test, score = 0.013541666666666667
np.mean(results), mse, len(results) =  0.016985433818872703 tf.Tensor(39.854404, shape=(), dtype=float32) 1579
np.mean(results), mse, len(results) =  0.013541666666666667 tf.Tensor(38.352516, shape=(), dtype=float32) 720
Saving ./GE_QE_RBExCoItem_military_noW__


=== Iteration 11000 ===
np.mean(results), mse, len(results) =  0.2571 tf.Tensor(2835.978, shape=(), dtype=float32) 100
slice = key, score = 0.2571
np.mean(results), mse, len(results) =  0.30752374920835973 tf.Tensor(2500.1738, shape=(), dtype=float32) 1579
slice = train, score = 0.30752374920835973
np.mean(results), mse, len(results) =  0.2736388888888889 tf.Tensor(2530.456, shape=(), dtype=float32) 720
slice = test, score = 0.2736388888888889
np.mean(results), mse, len(results) =  0.30752374920835973 tf.Tensor(2500.1738, shape=(), dtype=float32) 1579
np.mean(results), mse, len(results) =  0.2736388888888889 tf.Tensor(2530.456, shape=(), dtype=float32) 720
Saving ./GE_QE_RBExCoItem_military_noW__0.3075_0.2736.npz ...

=== Iteration 12000 ===
np.mean(results), mse, len(results) =  0.2587 tf.Tensor(3182.1519, shape=(), dtype=float32) 100
slice = key, score = 0.2587
np.mean(results), mse, len(results) =  0.30473717542748574 tf.Tensor(2830.4607, shape=(), dtype=float32) 1579
slice = train

...

np.mean(results), mse, len(results) =  0.37694743508549716 tf.Tensor(114859.6, shape=(), dtype=float32) 1579
np.mean(results), mse, len(results) =  0.32493055555555556 tf.Tensor(114903.35, shape=(), dtype=float32) 720
Saving ./GE_QE_RBExCoItem_military_noW__0.3769_0.3249.npz ...

=== Iteration 120000 ===
np.mean(results), mse, len(results) =  0.30400000000000005 tf.Tensor(117824.914, shape=(), dtype=float32) 100
slice = key, score = 0.30400000000000005
np.mean(results), mse, len(results) =  0.3744521849271691 tf.Tensor(116553.945, shape=(), dtype=float32) 1579
slice = train, score = 0.3744521849271691
np.mean(results), mse, len(results) =  0.3201527777777778 tf.Tensor(116501.984, shape=(), dtype=float32) 720
slice = test, score = 0.3201527777777778

=== Iteration 121000 ===
np.mean(results), mse, len(results) =  0.29950000000000004 tf.Tensor(119779.08, shape=(), dtype=float32) 100
slice = key, score = 0.29950000000000004
np.mean(results), mse, len(results) =  0.3735212159594681 tf.Tens

np.mean(results), mse, len(results) =  0.3226527777777778 tf.Tensor(147813.62, shape=(), dtype=float32) 720
slice = test, score = 0.3226527777777778

=== Iteration 137000 ===
np.mean(results), mse, len(results) =  0.3 tf.Tensor(151617.12, shape=(), dtype=float32) 100
slice = key, score = 0.3
np.mean(results), mse, len(results) =  0.37385053831538945 tf.Tensor(149767.0, shape=(), dtype=float32) 1579
slice = train, score = 0.37385053831538945
np.mean(results), mse, len(results) =  0.32258333333333333 tf.Tensor(149936.92, shape=(), dtype=float32) 720
slice = test, score = 0.32258333333333333

=== Iteration 138000 ===
np.mean(results), mse, len(results) =  0.3062 tf.Tensor(153445.31, shape=(), dtype=float32) 100
slice = key, score = 0.3062
np.mean(results), mse, len(results) =  0.3748511716276124 tf.Tensor(151942.6, shape=(), dtype=float32) 1579
slice = train, score = 0.3748511716276124
np.mean(results), mse, len(results) =  0.32338888888888884 tf.Tensor(151994.39, shape=(), dtype=float32)


=== Iteration 155000 ===
np.mean(results), mse, len(results) =  0.3024 tf.Tensor(189831.16, shape=(), dtype=float32) 100
slice = key, score = 0.3024
np.mean(results), mse, len(results) =  0.37738442051931603 tf.Tensor(187985.86, shape=(), dtype=float32) 1579
slice = train, score = 0.37738442051931603
np.mean(results), mse, len(results) =  0.32375000000000004 tf.Tensor(188008.83, shape=(), dtype=float32) 720
slice = test, score = 0.32375000000000004

=== Iteration 156000 ===
np.mean(results), mse, len(results) =  0.3012 tf.Tensor(191558.36, shape=(), dtype=float32) 100
slice = key, score = 0.3012
np.mean(results), mse, len(results) =  0.37341355288157063 tf.Tensor(189848.73, shape=(), dtype=float32) 1579
slice = train, score = 0.37341355288157063
np.mean(results), mse, len(results) =  0.3171527777777778 tf.Tensor(189854.62, shape=(), dtype=float32) 720
slice = test, score = 0.3171527777777778

=== Iteration 157000 ===
np.mean(results), mse, len(results) =  0.29679999999999995 tf.Tensor


=== Iteration 173000 ===
np.mean(results), mse, len(results) =  0.29900000000000004 tf.Tensor(230072.94, shape=(), dtype=float32) 100
slice = key, score = 0.29900000000000004
np.mean(results), mse, len(results) =  0.3734008866371122 tf.Tensor(228090.58, shape=(), dtype=float32) 1579
slice = train, score = 0.3734008866371122
np.mean(results), mse, len(results) =  0.32084722222222223 tf.Tensor(227927.2, shape=(), dtype=float32) 720
slice = test, score = 0.32084722222222223

=== Iteration 174000 ===
np.mean(results), mse, len(results) =  0.29729999999999995 tf.Tensor(232141.55, shape=(), dtype=float32) 100
slice = key, score = 0.29729999999999995
np.mean(results), mse, len(results) =  0.3735528815706143 tf.Tensor(230353.75, shape=(), dtype=float32) 1579
slice = train, score = 0.3735528815706143
np.mean(results), mse, len(results) =  0.31975 tf.Tensor(230460.19, shape=(), dtype=float32) 720
slice = test, score = 0.31975

=== Iteration 175000 ===
np.mean(results), mse, len(results) =  0.30

np.mean(results), mse, len(results) =  0.3695566814439519 tf.Tensor(268280.1, shape=(), dtype=float32) 1579
slice = train, score = 0.3695566814439519
np.mean(results), mse, len(results) =  0.3167638888888889 tf.Tensor(268198.97, shape=(), dtype=float32) 720
slice = test, score = 0.3167638888888889

=== Iteration 191000 ===
np.mean(results), mse, len(results) =  0.3041 tf.Tensor(273097.66, shape=(), dtype=float32) 100
slice = key, score = 0.3041
np.mean(results), mse, len(results) =  0.3794806839772008 tf.Tensor(270770.7, shape=(), dtype=float32) 1579
slice = train, score = 0.3794806839772008
np.mean(results), mse, len(results) =  0.3249722222222223 tf.Tensor(270754.2, shape=(), dtype=float32) 720
slice = test, score = 0.3249722222222223
np.mean(results), mse, len(results) =  0.3794806839772008 tf.Tensor(270770.7, shape=(), dtype=float32) 1579
np.mean(results), mse, len(results) =  0.3249722222222223 tf.Tensor(270754.2, shape=(), dtype=float32) 720
Saving ./GE_QE_RBExCoItem_military_noW

In [ ]:
+InnerDim + E + H

In [14]:
R = np.load('./R9650test.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=False
)

Best det =  0.0
Current de =  0.0
100 6654 2895


In [19]:
do(ctx, f"MsMarco_")

/var/tmp/ipykernel_2826/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

MemoryError: Unable to allocate 3.86 GiB for an array with shape (6654, 77897) and data type int64

In [18]:
do_RBE(ctx, f"MsMarco_", N=300000, kmeans=False, top=False, random=False)

/var/tmp/ipykernel_2826/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [17]:
gc.collect()

0